# FloodShift AI — End-to-End Decision Demo

This notebook demonstrates an **illustrative** San Antonio storm scenario: eight threatened locations, changing rainfall, one location-verified visual observation, and limited response resources. Synthetic inputs are never presented as real observations.

## 1. Load the backend and scenario

Run from the repository root after `python -m pip install --no-deps --no-build-isolation -e .`.

In [1]:
import json
from pathlib import Path
from IPython.display import JSON, Markdown, display

from floodshift.evaluation import compare_ranking_baselines
from floodshift.infrastructure import load_locations
from floodshift.pipeline import process_storm_update
from floodshift.vision import QwenVisionClient

root = Path.cwd()
if not (root / 'examples' / 'storm_scenario.json').is_file():
    root = root.parent
scenario = json.loads((root / 'examples' / 'storm_scenario.json').read_text())
locations = load_locations(root / 'data' / 'sample_locations.json')
print(scenario['scenario_name'])
print(scenario['data_policy'])

Illustrative San Antonio escalating storm
All rainfall, visual observations, and urgency labels in this file are synthetic demo inputs.


## 2. Optional GB10 health check

Run only when the vLLM server is reachable. The deterministic backend does not fail when Qwen is unavailable.

In [2]:
# QwenVisionClient().healthcheck()

## 3. Replay two storm updates

The second step adds a geographically matched visual observation. Historical Houston imagery must remain unlocated unless valid metadata ties it to a risk location.

In [3]:
results = []
previous = []
for step in scenario['steps']:
    rainfall = [
        {**item, 'timestamp': step['timestamp'], 'provenance': 'synthetic_demo',
         'source_name': 'Illustrative storm replay'}
        for item in step['rainfall']
    ]
    result = process_storm_update(
        rainfall, step['observations'], scenario['resources'],
        locations=locations, previous_assessments=previous
    )
    results.append(result)
    previous = result.risk_assessments

for index, result in enumerate(results, start=1):
    rows = ['| Rank | Location | Score | Severity | Change |', '|---:|---|---:|---|---:|']
    for item in result.risk_assessments[:5]:
        change = '—' if item.change_from_previous is None else f'{item.change_from_previous:+.1f}'
        rows.append(f'| {item.rank} | {item.location_name} | {item.risk_score:.1f} | {item.severity} | {change} |')
    display(Markdown(f'### Storm step {index}\n' + '\n'.join(rows)))

### Storm step 1
| Rank | Location | Score | Severity | Change |
|---:|---|---:|---|---:|
| 1 | New Sulphur Springs E of Beck | 33.0 | moderate | — |
| 2 | Pinn Rd 1320 ft S of West Commerce | 32.4 | moderate | — |
| 3 | Cougar Way Fort Sam Houston | 31.3 | moderate | — |
| 4 | Leslie Rd 3200 ft SW of Braun Rd | 30.9 | moderate | — |
| 5 | Pfeil Rd at Bluebell Ridge | 30.1 | moderate | — |

### Storm step 2
| Rank | Location | Score | Severity | Change |
|---:|---|---:|---|---:|
| 1 | Pinn Rd 1320 ft S of West Commerce | 63.8 | high | +31.4 |
| 2 | New Sulphur Springs E of Beck | 48.0 | moderate | +15.0 |
| 3 | Elm Creek at Jones Maltsberger | 47.7 | moderate | +18.4 |
| 4 | Hausman Rd 4200 ft W of IH-10 | 45.3 | moderate | +15.7 |
| 5 | Cougar Way Fort Sam Houston | 44.8 | moderate | +13.5 |

## 4. Inspect allocated and deferred actions

In [4]:
final = results[-1]
rows = ['| Status | Location | Action | Priority | Required resource |', '|---|---|---|---:|---|']
for item in final.recommendations:
    rows.append(f'| {item.allocation_status} | {item.location_name} | {item.action} | {item.priority_score:.1f} | {item.required_resource} |')
display(Markdown('\n'.join(rows)))

| Status | Location | Action | Priority | Required resource |
|---|---|---|---:|---|
| allocated | Pinn Rd 1320 ft S of West Commerce | road_closure_review | 75.8 | traffic_units + human_reviewers |
| allocated | Pinn Rd 1320 ft S of West Commerce | prioritize_debris_removal | 71.8 | debris_crews |
| allocated | Pinn Rd 1320 ft S of West Commerce | inspect_drainage_infrastructure | 69.8 | inspection_teams |
| allocated | New Sulphur Springs E of Beck | inspect_drainage_infrastructure | 54.0 | inspection_teams |
| allocated | New Sulphur Springs E of Beck | request_human_verification | 54.0 | human_reviewers |
| deferred_resource_limit | Elm Creek at Jones Maltsberger | inspect_drainage_infrastructure | 53.7 | inspection_teams |
| deferred_resource_limit | Elm Creek at Jones Maltsberger | request_human_verification | 53.7 | human_reviewers |
| deferred_resource_limit | Hausman Rd 4200 ft W of IH-10 | inspect_drainage_infrastructure | 51.3 | inspection_teams |
| deferred_resource_limit | Hausman Rd 4200 ft W of IH-10 | request_human_verification | 51.3 | human_reviewers |
| deferred_resource_limit | Cougar Way Fort Sam Houston | inspect_drainage_infrastructure | 50.8 | inspection_teams |
| deferred_resource_limit | Cougar Way Fort Sam Houston | request_human_verification | 50.8 | human_reviewers |
| deferred_resource_limit | Leslie Rd 3200 ft SW of Braun Rd | inspect_drainage_infrastructure | 49.5 | inspection_teams |
| deferred_resource_limit | Leslie Rd 3200 ft SW of Braun Rd | request_human_verification | 49.5 | human_reviewers |
| deferred_resource_limit | Pfeil Rd at Bluebell Ridge | inspect_drainage_infrastructure | 45.4 | inspection_teams |
| deferred_resource_limit | Pfeil Rd at Bluebell Ridge | request_human_verification | 45.4 | human_reviewers |
| deferred_resource_limit | Dreamland at RR Crossing | inspect_drainage_infrastructure | 42.9 | inspection_teams |
| deferred_resource_limit | Dreamland at RR Crossing | request_human_verification | 42.9 | human_reviewers |

## 5. Compare against simpler baselines

The labels are human-authored for this illustrative scenario. Results measure scenario prioritization—not real-world damage reduction.

In [5]:
last_step = scenario['steps'][-1]
last_rainfall = [
    {**item, 'timestamp': last_step['timestamp'], 'provenance': 'synthetic_demo'}
    for item in last_step['rainfall']
]
evaluation = compare_ranking_baselines(
    locations, last_rainfall, last_step['observations'],
    scenario['scenario_relevance'], k=3
)
JSON(evaluation)

<IPython.core.display.JSON object>

## Judge takeaway

Rainfall alone misses the visually confirmed crossing in the top three. FloodShift fuses the location-verified evidence, ranks it first, and allocates limited traffic, inspection, debris, emergency, and human-review capacity while keeping deferred actions visible.